# 02. Feature Engineering & Preprocessing

This notebook covers the feature engineering and preprocessing steps:
- Handling outliers by clipping.
- Creating calendar features (`hour`, `dayofweek`, `month`, `is_weekend`).
- Creating Fourier features (`sin_24`, `cos_24`, `sin_168`, `cos_168`).
- Creating lag features (`OT_lag_1`, `OT_lag_2`, etc.) and rolling statistics.
- Splitting into train/validation/test chronologically (70% / 15% / 15%).
- Performing Z-score normalization (fitting only on the training set to prevent leakage).
- Saving the processed splits to `data/processed/`.

In [1]:
import sys
sys.path.append('../')

import os
import numpy as np
import pandas as pd

from src.data_loader import load_data, split_data
from src.features import handle_outliers, create_calendar_features, create_fourier_features, create_lag_features, create_rolling_features, scale_datasets

## 1. Load Raw Data

In [2]:
filepath = "../data/raw/ETTm2.csv"
df = load_data(filepath)
df.head()

,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,41.130001,12.481,36.535999,9.355,4.424,1.311,38.661999
1,2016-07-01 00:15:00,39.622002,11.309,35.543999,8.551,3.209,1.258,38.223000
2,2016-07-01 00:30:00,38.868000,10.555,34.365002,7.586,4.435,1.258,37.344002
3,2016-07-01 00:45:00,35.518002,9.214,32.569000,8.712,4.435,1.215,37.124001
4,2016-07-01 01:00:00,37.528000,10.136,33.936001,7.532,4.435,1.215,37.124001


## 2. Handle Outliers

In [3]:
cols = [c for c in df.columns if c != 'date']
df = handle_outliers(df, cols, method='iqr')
df.head()

,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,41.130001,12.481,36.535999,9.355,4.424,1.311,38.661999
1,2016-07-01 00:15:00,39.622002,11.309,35.543999,8.551,3.209,1.258,38.223000
2,2016-07-01 00:30:00,38.868000,10.555,34.365002,7.586,4.435,1.258,37.344002
3,2016-07-01 00:45:00,35.518002,9.214,32.569000,8.712,4.435,1.215,37.124001
4,2016-07-01 01:00:00,37.528000,10.136,33.936001,7.532,4.435,1.215,37.124001


## 3. Create Features

In [4]:
# Calendar features
df = create_calendar_features(df, 'date')

# Fourier features (periods of 24 hours and 168 hours)
df = create_fourier_features(df, 'date', periods=[24, 168])

# Lag features for the target column 'OT' (15-min data: 96 steps = 24h, 192 steps = 48h, 672 steps = 7d)
df = create_lag_features(df, ['OT'], lags=[1, 2, 3, 96, 192, 672])

# Rolling features for target column 'OT' (96 steps = 24h, 672 steps = 7d)
df = create_rolling_features(df, ['OT'], windows=[96, 672])

# Drop rows with NaN values resulting from lags and rolling statistics
df = df.dropna().reset_index(drop=True)
print("Shape after feature engineering and dropping NaNs:", df.shape)
df.head()

Shape after feature engineering and dropping NaNs: (69008, 26)


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT,hour,dayofweek,...,OT_lag_1,OT_lag_2,OT_lag_3,OT_lag_96,OT_lag_192,OT_lag_672,OT_roll_mean_96,OT_roll_std_96,OT_roll_mean_672,OT_roll_std_672
0,2016-07-08 00:00:00,44.731998,13.905,42.433998,11.741,1.311,0.0,38.223000,0,4,...,38.442501,38.442501,38.661999,33.608501,33.168999,38.661999,38.836089,5.511782,31.650179,4.629515
1,2016-07-08 00:15:00,42.470001,12.063,40.127998,9.730,1.311,0.0,38.002998,0,4,...,38.223000,38.442501,38.442501,33.608501,33.168999,38.223000,38.884157,5.485773,31.649526,4.628555
2,2016-07-08 00:30:00,40.963001,11.141,39.699001,9.436,1.215,0.0,38.002998,0,4,...,38.002998,38.223000,38.442501,33.608501,33.168999,37.344002,38.929933,5.459559,31.649199,4.628097
3,2016-07-08 00:45:00,38.533001,11.057,37.286999,9.757,0.000,0.0,38.002998,0,4,...,38.002998,38.002998,38.223000,33.608501,33.168999,37.124001,38.975709,5.432829,31.650179,4.629376
4,2016-07-08 01:00:00,41.799999,12.565,40.852001,10.937,0.000,0.0,38.002998,1,4,...,38.002998,38.002998,38.002998,33.608501,33.168999,37.124001,39.021485,5.405575,31.651487,4.631048


## 4. Chronological Splitting

In [5]:
train_df, val_df, test_df = split_data(df, train_ratio=0.7, val_ratio=0.15)
print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (48305, 26)
Val shape: (10351, 26)
Test shape: (10352, 26)


## 5. Normalize Data (Z-score)

In [6]:
# Normalizing all columns except 'date'
numeric_cols = [col for col in train_df.columns if col != 'date']

train_scaled, val_scaled, test_scaled, mean, std = scale_datasets(train_df, val_df, test_df, numeric_cols)
print("Mean values used for scaling:", mean.head(5))
print("Offset values (Std) used for scaling:", std.head(5))

Mean values used for scaling: HUFL    39.444642
HULL    10.353522
MUFL    42.852380
MULL     9.446230
LUFL    -1.206391
dtype: float64
Offset values (Std) used for scaling: HUFL    9.204573
HULL    5.740936
MUFL    8.002656
MULL    3.970003
LUFL    5.627506
dtype: float64


## 6. Save Processed Datasets

In [7]:
os.makedirs('../data/processed', exist_ok=True)
train_scaled.to_csv('../data/processed/train.csv', index=False)
val_scaled.to_csv('../data/processed/val.csv', index=False)
test_scaled.to_csv('../data/processed/test.csv', index=False)
print("Saved scaled splits successfully.")

Saved scaled splits successfully.
